In [ ]:
# File: 03_farid_tuning_best.ipynb
# Author: Farid
# Task: Comparison + Tuning + Best Model
# ------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("FARID - Comparison + Tuning + Best Model")
print("=" * 60)

# ----------------------------------------------
# 1. Load Data

df = pd.read_csv('../../1_Data/cleaned_data.csv')
df = df.drop('Property_ID', axis=1)
df['Is_Sold'] = df['Is_Sold'].map({'Yes': 1, 'No': 0})
df_encoded = pd.get_dummies(df, columns=['Location', 'Property_Type'], drop_first=True)


X = df_encoded.drop(['Price_EGP', 'Is_Sold'], axis=1)
y = df_encoded['Price_EGP']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling (Linear Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ----------------------------------------------
# 2. Load Previous Results

results_youssef = pd.read_csv('results_youssef.csv')
results_adel = pd.read_csv('results_adel.csv')
all_results = pd.concat([results_youssef, results_adel], ignore_index=True)

print("\n--------- All Models (Before Tuning) ----------")
print(all_results)

# ----------------------------------------------
# 3. Tuning XGBoost

print("\n" + "=" * 60)
print("TUNING XGBoost")
print("=" * 60)

param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7]
}

xgb = XGBRegressor(random_state=42)
grid_xgb = GridSearchCV(xgb, param_grid_xgb, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid_xgb.fit(X_train, y_train)

print(f"\nBest Params: {grid_xgb.best_params_}")
print(f"Best CV Score: {grid_xgb.best_score_:.4f}")

best_xgb = grid_xgb.best_estimator_
best_xgb_pred = best_xgb.predict(X_test)

best_xgb_mae = mean_absolute_error(y_test, best_xgb_pred)
best_xgb_rmse = np.sqrt(mean_squared_error(y_test, best_xgb_pred))
best_xgb_r2 = r2_score(y_test, best_xgb_pred)

print(f"\n=== Tuned XGBoost ===")
print(f"MAE:  {best_xgb_mae:,.2f}")
print(f"RMSE: {best_xgb_rmse:,.2f}")
print(f"R²:   {best_xgb_r2:.4f}")

# ----------------------------------------------
# 4. Tuning Random Forest

print("\n" + "=" * 60)
print("TUNING Random Forest")
print("=" * 60)

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5]
}

rf = RandomForestRegressor(random_state=42)
grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid_rf.fit(X_train, y_train)

print(f"\nBest Params: {grid_rf.best_params_}")
print(f"Best CV Score: {grid_rf.best_score_:.4f}")

best_rf = grid_rf.best_estimator_
best_rf_pred = best_rf.predict(X_test)

best_rf_mae = mean_absolute_error(y_test, best_rf_pred)
best_rf_rmse = np.sqrt(mean_squared_error(y_test, best_rf_pred))
best_rf_r2 = r2_score(y_test, best_rf_pred)

print(f"\n=== Tuned Random Forest ===")
print(f"MAE:  {best_rf_mae:,.2f}")
print(f"RMSE: {best_rf_rmse:,.2f}")
print(f"R²:   {best_rf_r2:.4f}")

# ----------------------------------------------
# 5. Train Linear Regression + Decision Tree

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

dt = DecisionTreeRegressor(random_state=42, max_depth=10)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

dt_mae = mean_absolute_error(y_test, dt_pred)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
dt_r2 = r2_score(y_test, dt_pred)

# ----------------------------------------------
# 6. Final Comparison

final_results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest', 'XGBoost',
              'Tuned Random Forest', 'Tuned XGBoost'],
    'MAE': [lr_mae, dt_mae, all_results.loc[2, 'MAE'], all_results.loc[3, 'MAE'],
            best_rf_mae, best_xgb_mae],
    'RMSE': [lr_rmse, dt_rmse, all_results.loc[2, 'RMSE'], all_results.loc[3, 'RMSE'],
             best_rf_rmse, best_xgb_rmse],
    'R2': [lr_r2, dt_r2, all_results.loc[2, 'R2'], all_results.loc[3, 'R2'],
           best_rf_r2, best_xgb_r2]
})

final_results = final_results.sort_values('R2', ascending=False).reset_index(drop=True)
print("\n=== FINAL COMPARISON ===")
print(final_results)

final_results.to_csv('final_comparison.csv', index=False)
print("\n final_comparison.csv saved")

# ----------------------------------------------
# 7. Save Best Model

best_model_name = final_results.iloc[0]['Model']
print("\n" + "=" * 60)
print(f"BEST MODEL: {best_model_name}")
print("=" * 60)

if best_model_name == 'Linear Regression':
    joblib.dump(lr, 'best_model.pkl')
    joblib.dump(scaler, 'scaler.pkl')
    print("best_model.pkl (Linear Regression) saved")
    print("scaler.pkl saved")

elif best_model_name == 'Decision Tree':
    joblib.dump(dt, 'best_model.pkl')
    print("best_model.pkl (Decision Tree) saved")

elif 'XGBoost' in best_model_name:
    joblib.dump(best_xgb, 'best_model.pkl')
    print("best_model.pkl (XGBoost) saved")

elif 'Random Forest' in best_model_name:
    joblib.dump(best_rf, 'best_model.pkl')
    print("best_model.pkl (Random Forest) saved")

# Save feature columns
joblib.dump(X.columns.tolist(), 'feature_columns.pkl')
print("feature_columns.pkl saved")

# Save model info
model_info = {
    'model_name': best_model_name,
    'features': X.columns.tolist(),
    'target': 'Price_EGP',
    'needs_scaling': best_model_name == 'Linear Regression'
}

joblib.dump(model_info, 'model_info.pkl')
print("model_info.pkl saved")

# ----------------------------------------------
# 8. Verify

print("\n" + "=" * 60)
print("VERIFICATION")
print("=" * 60)

import os
files = ['best_model.pkl', 'scaler.pkl', 'feature_columns.pkl', 'model_info.pkl']
for f in files:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f" {f} ({size} bytes)")
    else:
        print(f" {f} - NOT FOUND")

print("\n" + "=" * 60)
print("FARID - DONE WITH COMPARISON + TUNING + BEST MODEL")
print("=" * 60)

FARID - Comparison + Tuning + Best Model

=== All Models (Before Tuning) ===
               Model           MAE          RMSE        R2
0  Linear Regression  3.157737e+06  3.866100e+06 -0.033264
1      Decision Tree  3.583859e+06  4.619457e+06 -0.475187
2      Random Forest  3.309401e+06  4.004579e+06 -0.108610
3            XGBoost  3.759825e+06  4.410333e+06 -0.344646

TUNING XGBoost
Fitting 5 folds for each of 27 candidates, totalling 135 fits

Best Params: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}
Best CV Score: -0.1351

=== Tuned XGBoost ===
MAE:  3,231,461.75
RMSE: 3,886,699.34
R²:   -0.0443

TUNING Random Forest
Fitting 5 folds for each of 12 candidates, totalling 60 fits
